In [2]:
import pandas as pd

# 定义通用大文件读取函数
def read_big_csv(file_path, chunk_size=10000):
    chunk_list = []
    # 分块读取+容错编码
    for chunk in pd.read_csv(file_path, chunksize=chunk_size,
                             encoding="utf-8-sig", encoding_errors="ignore"):
        chunk_list.append(chunk)
    df = pd.concat(chunk_list, ignore_index=True)
    return df

# 读取四份原始数据
df_users = read_big_csv("./data/raw/users.csv")
df_products = read_big_csv("./data/raw/products.csv")
df_browses = read_big_csv("./data/raw/browses.csv")
df_orders = read_big_csv("./data/raw/orders.csv")

In [6]:
# 时间转换
df_orders["created_at"] = pd.to_datetime(df_orders["created_at"])
max_date = df_orders["created_at"].max()

# 计算RFM
rfm = df_orders.groupby("user_id").agg(
    R=("created_at", lambda x: (max_date - x.max()).days),
    F=("id", "count"),
    M=("quantity", "sum")
).reset_index()
print("RFM指标计算完成，预览：")
print(rfm.head(10))

RFM指标计算完成，预览：
   user_id   R   F   M
0        1  13  33  48
1        2  10  29  44
2        3   3  40  58
3        4   7  55  74
4        5  48  41  58
5        6  18  49  59
6        7   1  38  54
7        8  11  40  58
8        9   6  46  58
9       10  10  54  66


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 标准化
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["R", "F", "M"]])

# Kmeans四分类
kmeans = KMeans(n_clusters=4, random_state=42)
rfm["cluster"] = kmeans.fit_predict(rfm_scaled)

# 统计各类用户均值
print("===== 用户分层结果 =====")
print(rfm.groupby("cluster")[["R", "F", "M"]].mean())

===== 用户分层结果 =====
                  R          F          M
cluster                                  
0        391.460446   1.453823   2.089852
1         62.013420   2.761938   3.986939
2         11.526884  42.007102  60.665433
3        210.049921   2.175446   3.151749
